# 05 - Chronological Train / Validation / Test Split

Per the decision logged after EDA, we use a **chronological, proportion based split** rather than a random split or a strict calendar year cutoff - the 2018 data is too thin/seasonally narrow to hold out on its own.

**Update from the original two way plan:** we're adding an explicit **validation** split, not just train/test. Reason: our Initial Submission already planned to use Optuna for hyperparameter tuning, and running full cross validation inside every single Optuna trial (across 4 candidate models and many trials) is computationally expensive. A single fixed validation set lets each trial be evaluated once - much cheaper - while the test set stays completely untouched until final evaluation.

**All three splits stay chronologically ordered: train (earliest) → validation (middle) → test (most recent).** This keeps the "always evaluate on data that comes after what you trained on" principle intact at every step, not just at the final boundary.


## Setup

In [1]:
import pandas as pd
from pathlib import Path

INPUT_PATH = Path('../../../data/processed/orders_final.csv')
TRAIN_OUTPUT = Path('../../../data/processed/train.csv')
VAL_OUTPUT = Path('../../../data/processed/val.csv')
TEST_OUTPUT = Path('../../../data/processed/test.csv')

orders_df = pd.read_csv(INPUT_PATH)
orders_df['order date (DateOrders)'] = pd.to_datetime(orders_df['order date (DateOrders)'], errors='coerce')

print(f"Shape: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")
print(f"Date range: {orders_df['order date (DateOrders)'].min()} to {orders_df['order date (DateOrders)'].max()}")


Shape: 65,752 rows x 21 columns
Date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00


## 1. Sort chronologically and split into three chunks

In [2]:
VAL_FRACTION = 0.12   # middle chunk - used for Optuna trial evaluation
TEST_FRACTION = 0.18  # final chunk - untouched until final evaluation

orders_sorted = orders_df.sort_values('order date (DateOrders)').reset_index(drop=True)
n = len(orders_sorted)

test_start = int(n * (1 - TEST_FRACTION))
val_start = int(n * (1 - TEST_FRACTION - VAL_FRACTION))

train_df = orders_sorted.iloc[:val_start]
val_df = orders_sorted.iloc[val_start:test_start]
test_df = orders_sorted.iloc[test_start:]

for name, part in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f"{name}: {len(part):,} orders ({len(part)/n*100:.1f}%), "
          f"{part['order date (DateOrders)'].min()} to {part['order date (DateOrders)'].max()}")


Train: 46,026 orders (70.0%), 2015-01-01 00:00:00 to 2017-03-16 19:22:00
Validation: 7,890 orders (12.0%), 2017-03-16 19:43:00 to 2017-08-02 00:04:00
Test: 11,836 orders (18.0%), 2017-08-02 00:25:00 to 2018-01-31 23:38:00


## 2. Verify class balance holds across all three splits

Since this isn't a stratified split (chronological splits can't be, by definition), check that late/on time proportions stay reasonably close across train, validation, and test - EDA found no meaningful year over year drift, so this should hold.


In [3]:
for name, part in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    pct = part['Late_delivery_risk'].value_counts(normalize=True).round(4) * 100
    print(f"{name} class balance:")
    print(pct)
    print()


Train class balance:
Late_delivery_risk
1    54.85
0    45.15
Name: proportion, dtype: float64

Validation class balance:
Late_delivery_risk
1    54.23
0    45.77
Name: proportion, dtype: float64

Test class balance:
Late_delivery_risk
1    55.14
0    44.86
Name: proportion, dtype: float64



**What we found:**

**Class balance holds tightly across all three splits - the largest deviation from the overall 54.8%/45.2% split is under 1 percentage point:**

| Split | Late | Not-late |
|---|---|---|
| Train | 54.85% | 45.15% |
| Validation | 54.23% | 45.77% |
| Test | 55.14% | 44.86% |

This is a good, clean confirmation of the EDA finding that there's no meaningful year over year drift in late delivery rate - even the test set (the most recent slice, Aug 2017–Jan 2018, including that thin January 2018 tail) stays within about half a percentage point of the overall average. No flag needed here; the chronological split is behaving exactly as expected given what EDA predicted.

**Split sizes came out as planned:** Train 46,026 orders (70.0%, Jan 2015–Mar 2017), Validation 7,890 orders (12.0%, Mar–Aug 2017), Test 11,836 orders (18.0%, Aug 2017–Jan 2018) - clean, non overlapping, chronologically ordered date ranges confirm the split logic worked correctly.


## 3. Save all three splits

In [4]:
for path in [TRAIN_OUTPUT, VAL_OUTPUT, TEST_OUTPUT]:
    path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_csv(TRAIN_OUTPUT, index=False)
val_df.to_csv(VAL_OUTPUT, index=False)
test_df.to_csv(TEST_OUTPUT, index=False)

print(f"Train saved to: {TRAIN_OUTPUT.resolve()}")
print(f"Validation saved to: {VAL_OUTPUT.resolve()}")
print(f"Test saved to: {TEST_OUTPUT.resolve()}")


Train saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\train.csv
Validation saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\val.csv
Test saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\test.csv


**`DECISION_LOG.md`:** added an explicit validation split (train ~70% / validation ~12% / test ~18%, all chronologically ordered) rather than train/test alone - driven by the compute cost of running cross validation inside every Optuna trial across 4 candidate models. Test set remains completely held out until final evaluation; validation set is used only for Optuna trial scoring during tuning.

**How this connects to modelling stages:**
- **Stage 5/6 (baseline & model development):** train on `train.csv`, do initial comparison against `val.csv`.
- **Stage 8 (Optuna tuning):** each trial fits on `train.csv`, scores on `val.csv` - fast, single evaluation per trial rather than k-fold.
- **Stage 7 (final evaluation):** `test.csv` is opened exactly once, after the final model/hyperparameters are locked in - never used for any tuning decision.
- **Optional, if time permits later:** `TimeSeriesSplit` cross validation on `train.csv` alone (not touching val or test) can still be run as an extra robustness check on the final chosen model, without changing this core train/val/test structure.

**Next stage:** with `train.csv`, `val.csv`, and `test.csv` ready, Stage 4 (Feature Engineering) and Stage 5/6 (Baseline & Model Development) can begin.
